# Bias in data analysis: checking the bike shop's plan

**Student notebook** · SDEV2501 Lesson 19

Name:

The owner of the Edmonton bike shop from Lessons 22-23 (fictional) is back with a decision to make:

> *"I'm putting together next year's bike order and the supplier needs it by Friday. Our spring survey says customers want e-bikes, every e-bike model we carry has sold well, and XS and S frames barely sell. So I'm doubling the e-bike order and dropping XS and S. Can you check my numbers before I sign?"*

Each of the owner's reasons comes from real data, and the arithmetic behind each one is correct. The question in this case study is a different one: could that data give a fair answer in the first place? Each of the four kinds of bias on the "Types of Bias" slide shows up once.

| Owner's reason | Data | Bias to look for | Section |
|---|---|---|---|
| "Customers want e-bikes" | `customer_survey.csv` | Sampling bias: who answered | 1 |
| "Customers want e-bikes" | `customer_survey.csv` | Measurement bias: how the answer was recorded | 2 |
| "XS and S frames barely sell" | `frame_size_weeks.csv` | Historical bias: past decisions recorded in the data | 3 |
| "Every e-bike model we carry has sold well" | `ebike_models.csv` | Survivorship bias: what's missing from the list | 4 |

Section 5 corrects the survey estimate and records each bias in a bias log, and section 6 is your reply to the owner.

The files are in this folder: `customer_survey.csv` (400 responses, April to August 2026), `frame_size_weeks.csv` (one row per frame size per week, November 2025 to October 2026), `ebike_models.csv` (every e-bike model the shop has stocked since 2021) and `bike_shop_orders.csv` (the order file from Lessons 22-23).

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt

## 1. Who answered the survey?

The survey ran from April to August 2026 and went out two ways: by email to the shop's newsletter list, and on a tablet at the till. Its e-bike question was "How interested are you in buying an e-bike in the next 12 months?", answered from 1 (not at all) to 5 (very). The owner reads the result as what customers want, so the first check is whether the people who answered look like the shop's customers.

### 1.1 Load the survey

Load `customer_survey.csv` into a DataFrame called `survey` and show its shape and first few rows. Then add a True/False column called `interested` that is True when `ebike_interest` is 4 or 5, and print the share of respondents who are interested.

| You'll need | What it's for |
|---|---|
| `pd.read_csv(...)` | Reads the file into a DataFrame. |
| `.shape`, `.head()` | The (rows, columns) size and the first five rows. |
| `survey["..."] >= ...` | A comparison on a column gives a True/False Series, which you can store as a new column. |
| `.mean()` on a True/False column | True counts as 1 and False as 0, so the mean is the share of rows that are True. |

### 1.2 Compare respondents with customers

Load `bike_shop_orders.csv` into `orders`. Make a DataFrame called `customers` with one row per customer. Then build a DataFrame called `mix` with two columns: the share of survey respondents in each membership type (`survey`) and the share of customers in each membership type (`customers`).

| You'll need | What it's for |
|---|---|
| `.drop_duplicates(subset=...)` | Keeps the first row for each value of the column you name, so each customer appears once. |
| `.value_counts(normalize=...)` | Counts each value. Passing `normalize=True` gives shares that add up to 1 instead of counts. |
| `pd.DataFrame({"...": ..., "...": ...})` | Builds a table from Series, one column per key, lining rows up by their index labels. |

### 1.3 Chart the comparison

Draw a bar chart of `mix` with the two shares side by side for each membership type.

| You'll need | What it's for |
|---|---|
| `mix.plot(kind="bar")` | Draws a bar for each column in every row, side by side, with a legend. |
| `plt.title(...)`, `plt.xlabel(...)`, `plt.ylabel(...)` | Labels the chart and its axes. |

### 1.4 Interest by membership type

Find the share of respondents who are interested within each membership type.

| You'll need | What it's for |
|---|---|
| `.groupby("...")["..."].mean()` | Splits the rows by membership type and takes the mean of `interested` in each group, which is that group's share. |

Explain how the people who answered the survey differ from the shop's customers, and why that matters for the owner's claim that "customers want e-bikes".

*Your answer:*

## 2. How was the answer recorded?

The two channels didn't ask the question the same way. By email it's five buttons, with none selected. On the tablet it's a slider that customers drag to a number. The file also records `seconds_taken`, how long each person spent on the whole survey.

### 2.1 Answers by channel

Make a table of the share of each rating (1 to 5) within each channel.

| You'll need | What it's for |
|---|---|
| `pd.crosstab(..., ..., normalize=...)` | Counts every combination of two columns. Passing `normalize="index"` divides each row by its total, so each channel's row adds to 1. |

### 2.2 Time taken

Draw a histogram of `seconds_taken` with bins about 5 seconds wide.

| You'll need | What it's for |
|---|---|
| `plt.hist(..., bins=...)` | Draws a histogram. The values run from about 0 to 250 seconds, so 50 bins makes each one about 5 seconds wide. |

### 2.3 The fast responses

Make a DataFrame called `fast` with the responses that took under 20 seconds. Show how many there are in each channel and what rating they gave.

| You'll need | What it's for |
|---|---|
| a boolean mask, `survey[survey["..."] < ...]` | Keeps only the rows where the condition is True. |
| `.value_counts()` | Counts each value in a column. |

Explain what you think happened with the fast responses, why this is measurement bias rather than sampling bias, and which way it pushes the share of interested customers.

*Your answer:*

## 3. Do XS and S frames really barely sell?

`frame_size_weeks.csv` has one row per frame size per week. `in_stock` is True when at least one bike in that size was on the shop floor that week, and `units_sold` counts adult bikes and commuter e-bikes sold in that size. Cargo e-bikes and kids' bikes come in one size and aren't included. The stock levels come from the previous buyer's orders.

### 3.1 Total sales by size

Load the file into `frames` and total `units_sold` for each frame size, in size order from XS to XL.

| You'll need | What it's for |
|---|---|
| `.groupby("...")["..."].sum()` | Adds up a column within each group. |
| `.reindex([...])` | Puts the rows in the order you list. `groupby` sorts the sizes alphabetically (L, M, S, XL, XS). |

### 3.2 Sales per week in stock

Make a DataFrame called `by_size` with, for each frame size, the total units sold and the number of weeks the size was in stock. Add a column `units_per_week` that divides units by weeks in stock.

| You'll need | What it's for |
|---|---|
| `.groupby("...").agg(name=("...", "..."), ...)` | Named aggregation: each `name=(column, function)` makes one output column, so one call can sum two columns. |
| summing a True/False column | `"sum"` on `in_stock` counts the True weeks. |
| dividing one column by another | Works row by row and gives a new column. |

### 3.3 Chart it

Draw a bar chart of units sold per week in stock for each frame size.

| You'll need | What it's for |
|---|---|
| `plt.bar(..., ...)` | One bar per frame size; the first argument is the labels, the second the heights. |

Explain why the yearly totals make XS and S frames look unpopular, what this data can't show about customers who wanted a small frame, and why this is historical bias.

*Your answer:*

## 4. Has every e-bike model sold well?

The owner's claim comes from the e-bike models on the shop floor today. `ebike_models.csv` lists every e-bike model the shop has stocked since 2021, including the ones it dropped, with the units sold in each model's first year.

### 4.1 Load the models

Load `ebike_models.csv` into `models` and show the whole table.

| You'll need | What it's for |
|---|---|
| `pd.read_csv(...)` | Reads the file. With only 11 rows, showing `models` prints all of it. |

### 4.2 Current models vs all models

Find the number of models and the mean, minimum and maximum first-year units for current and discontinued models. Then find the mean first-year units across all models.

| You'll need | What it's for |
|---|---|
| `.groupby("...")["..."].agg([...])` | Applies several functions to each group at once when you pass a list of names, such as `"count"` and `"mean"`. |
| `.mean()` | The mean of the whole column. |

Explain what the owner's "every e-bike model we carry has sold well" leaves out, and why this is survivorship bias.

*Your answer:*

## 5. Reducing the bias

Finding a bias is half the job. The other half is doing something about it: correcting the number where the data allows it, and writing down what you did and what you couldn't fix.

### 5.1 Correct the survey estimate

Correct the share of interested customers in two steps, printing the result after each one.

First, make a DataFrame called `clean` without the responses that took under 20 seconds, and find its share of interested respondents.

Second, reweight `clean` to match the customers: find the share interested within each membership type, multiply each by that type's share of customers (the `customers` column of `mix`), and add the results up. This is the share you'd expect if the survey had the same membership mix as the shop's customers.

| You'll need | What it's for |
|---|---|
| a boolean mask, `survey[survey["..."] >= ...]` | Keeps the rows that pass the condition. |
| `.groupby("...")["..."].mean()` | Share interested within each membership type. |
| `series_a * series_b` | Multiplies two Series by matching index labels, so Basic is multiplied by Basic whatever order the rows are in. |
| `.sum()` | Adds up the weighted shares. |

Explain how the estimate changed at each step, and what the reweighted number still assumes.

*Your answer:*

### 5.2 Bias log

Complete the bias log. For each bias, say where it came from in this case, how it changed the owner's conclusion, and one way to reduce it next time.

| Bias | Where it came from | How it changed the owner's conclusion | How to reduce it next time |
|---|---|---|---|
| Sampling | | | |
| Measurement | | | |
| Historical | | | |
| Survivorship | | | |

## 6. Reply to the owner

Explain, in a short reply to the owner (four to six sentences in plain language), what the data does and doesn't support about each of the three reasons, and what you'd suggest before signing the order on Friday.

*Your answer:*